## *Tools

To know more about available tools, Visit at [LangChain Documentation - Tools](https://docs.langchain.com/oss/python/langchain/tools)

### **Built-in Tools

In [3]:
from langchain_classic.tools import DuckDuckGoSearchRun, ShellTool
import warnings

warnings.filterwarnings("ignore")

search_tools = DuckDuckGoSearchRun()

search_tools.invoke("Whats the Current date.")

"See today's exactdateandtimeupdated live in every format — ISO 8601, Unix timestamp, Juliandate, day of year, week number, moon phase, and more. 4 hours ago ·Find today’sdate,currentday of the week, and livetimeinstantly. Automatically updates daily with week number and year progress. 23 hours ago ·World Time Server showscurrentlocal time anddatein cities and countries in all time zones, adjusted for Daylight Saving Time rules automatically. 23 hours ago ·Our free online today'sdatetool displays thecurrentdateand time in multiple formats, timezones, and calendars. Perfect fordatereferences, scheduling, time zone conversions, and getting accurate timestamps. 1 day ago ·Check thecurrentdateand time anywhere in the world. Accurate live clocks and time zones."

In [4]:
shell = ShellTool()

shell.invoke("whoami")

Executing command:
 whoami


'prashant\n'

In [5]:
get_path = shell.invoke("pwd")
type(get_path)

Executing command:
 pwd


str

### **Manual Tools Creation

In [6]:
from langchain_core.tools import tool

@tool
def add(a:float, b:float)->float:
    """Add two numbers"""
    return a+b

In [7]:
get_result = add.invoke({'a':5,'b':6})

print(get_result)

11.0


In [8]:
## Print tool args

print(add.name)
print(add.args)
print(add.description)
print(add.args_schema.model_json_schema())

add
{'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
Add two numbers
{'description': 'Add two numbers', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}


In [9]:
search_tools.args_schema.model_json_schema()

{'description': 'Input for the DuckDuckGo search tool.',
 'properties': {'query': {'description': 'search query to look up',
   'title': 'Query',
   'type': 'string'}},
 'required': ['query'],
 'title': 'DDGInput',
 'type': 'object'}

### **Using StructuredTool

In [10]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [11]:
class AddInputs(BaseModel):

    a: float = Field(required=True, description="The first number to add")
    b: float = Field(required=True, description="The second number to add")

def _add(a: float, b:float)->float:
    return a+b

In [12]:
## Creating Structured tool

add_numbers = StructuredTool.from_function(
    func=_add,
    name="addition",
    description="Add two numbers",
    args_schema=AddInputs
)

In [13]:
res = add_numbers.invoke({'a':5,'b':6})

print(res)
print(add_numbers.name)
print(add_numbers.args)
print(add_numbers.args_schema.model_json_schema())
print(add_numbers.description)

11.0
addition
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'number'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'number'}}
{'properties': {'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'number'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'AddInputs', 'type': 'object'}
Add two numbers


### **Using the BaseTool

In [14]:
from langchain_core.tools import BaseTool
from typing import Type

# Args achema using PyDantic
class AddInputs(BaseModel):

    a: float = Field(required=True, description="The first number to add")
    b: float = Field(required=True, description="The second number to add")

class AdditionTool(BaseTool):

    name: str = "addition"
    description: str = "Add two numbers"

    args_schema: Type[BaseModel] = AddInputs

    def _run(self, a: float, b:float)->float:
        return a+b

In [15]:
addition_tool = AdditionTool()

result = addition_tool.invoke({'a':5,'b':6})

In [16]:
print(result)

11.0


### **Toolkit

In [17]:
@tool
def multiply(a:float, b:float)->float:
    """Multiply two numbers"""
    return a*b

@tool
def add(a:float, b:float)->float:
    """Add two numbers"""
    return a+b

In [18]:
## Toolkit class
class MathToolkit:
    def get_tools(self):
        return [multiply,add]
    
toolkit = MathToolkit()

tools = toolkit.get_tools()

for i,tool in enumerate(tools):
    print(f"Tool {i+1}:{tool.args_schema.model_json_schema()}")

Tool 1:{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}
Tool 2:{'description': 'Add two numbers', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}


## *Tool Calling

In [85]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

llm=ChatOllama(model='gpt-oss:120b-cloud')

In [21]:
llm_with_tools = llm.bind_tools([add,multiply])

In [27]:
result = llm_with_tools.invoke("Who are you?")

In [30]:
result = llm_with_tools.invoke("What's 5 multiplied by 9?")

In [31]:
result

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3-vl:235b-instruct-cloud', 'created_at': '2026-05-24T23:41:00.714892463Z', 'done': True, 'done_reason': 'stop', 'total_duration': 988349286, 'load_duration': None, 'prompt_eval_count': 227, 'prompt_eval_duration': None, 'eval_count': 25, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-vl:235b-instruct-cloud', 'model_provider': 'ollama'}, id='lc_run--019e5c5c-fdf3-7d13-bb13-a326be409c74-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 9}, 'id': 'd89966e0-29ac-4050-88d9-4b741371ac26', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 227, 'output_tokens': 25, 'total_tokens': 252})

Tool will be automatically called/initialized when the requirements occur.

In [32]:
query = HumanMessage("What's 5 multiplied by 9?")
message = [query]

message.append(result)

In [34]:
tool_result = multiply.invoke(result.tool_calls[0])
tool_result

ToolMessage(content='45.0', name='multiply', tool_call_id='d89966e0-29ac-4050-88d9-4b741371ac26')

In [35]:
message.append(tool_result)
message

[HumanMessage(content="What's 5 multiplied by 9?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3-vl:235b-instruct-cloud', 'created_at': '2026-05-24T23:41:00.714892463Z', 'done': True, 'done_reason': 'stop', 'total_duration': 988349286, 'load_duration': None, 'prompt_eval_count': 227, 'prompt_eval_duration': None, 'eval_count': 25, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-vl:235b-instruct-cloud', 'model_provider': 'ollama'}, id='lc_run--019e5c5c-fdf3-7d13-bb13-a326be409c74-0', tool_calls=[{'name': 'multiply', 'args': {'a': 5, 'b': 9}, 'id': 'd89966e0-29ac-4050-88d9-4b741371ac26', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 227, 'output_tokens': 25, 'total_tokens': 252}),
 ToolMessage(content='45.0', name='multiply', tool_call_id='d89966e0-29ac-4050-88d9-4b741371ac26')]

In [36]:
llm_with_tools.invoke(message).content

'5 multiplied by 9 equals 45.'

### Currency conversion tool creation

In [102]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

## Tool creation
@tool
def get_conversion_factor(base_currency:str, target_currency:str)-> float:

    """
    This function fetches a JSON with multiple values including the conversion rate between the base_currency and target_currency.
    """
    url = f"https://v6.exchangerate-api.com/v6/81554ec926c2a26d17463601/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    
    return response.json()

@tool
def convert_currency(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg])-> float:
    """
    In this, we get base_currency_value to convert using given conversion rate and return the converted amount.
    """
    return base_currency_value * conversion_rate

In [103]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1779667201,
 'time_last_update_utc': 'Mon, 25 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1779753601,
 'time_next_update_utc': 'Tue, 26 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.8242}

In [105]:
convert_currency.invoke({'base_currency_value': 32, 'conversion_rate': 95.8242})

3066.3744

In [106]:
llm

ChatOllama(model='gpt-oss:120b-cloud')

In [107]:
currency_conversion_tool = llm.bind_tools([get_conversion_factor,convert_currency])

In [108]:
message = [HumanMessage("What is the conversion rate between USD & INR, and return the amount in INR based on the conversion rate 32.4 USD? Use attached tool for final answer.")]
message

[HumanMessage(content='What is the conversion rate between USD & INR, and return the amount in INR based on the conversion rate 32.4 USD? Use attached tool for final answer.', additional_kwargs={}, response_metadata={})]

In [109]:
ai_message = currency_conversion_tool.invoke(message)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'ccab46fb-4f51-46aa-b9d4-a1579326e5d5',
  'type': 'tool_call'}]

In [ ]:
tool_msg = convert_currency.invoke(ai_message.tool_calls[0])

In [111]:
message.append(ai_message)